# Reproduction of manuscript Figure 4

Intermediate-coupling three-site calculation. The notebook first performs 100 independent random-order optimizations and identifies the two stationary-solution families reported in the manuscript, then calculates Present 1, Present 2, CMRT, and Förster dynamics.

In [ ]:
from pathlib import Path
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from QME import Config, run_system, uniqueness_scan_system
from manuscript_models import three_site_model

OUT = Path("results")
OUT.mkdir(exist_ok=True)

H = three_site_model(100.0, 100.0, 20.0, 20.0)
lambda_i = np.full(3, 100.0)
cfg = Config(temperature_k=77.0, tau_g_ps=0.05, nmax=10_000, dt_ps=1.0e-4)


In [ ]:
columns, scan, bases = uniqueness_scan_system(H, lambda_i, cfg, n_runs=100, seed_start=0)
np.savetxt(OUT / "Fig4_100seed_scan.dat", scan, header=" ".join(columns))

# Ordered diagonal energies of the two stationary families (cm^-1).
target1 = np.array([-38.076, 17.781, 80.296])
target2 = np.array([-40.296, 22.220, 78.076])
E = scan[:, 4:7]
i1 = int(np.argmin(np.linalg.norm(E - target1, axis=1)))
i2 = int(np.argmin(np.linalg.norm(E - target2, axis=1)))
seed1, seed2 = int(scan[i1, 1]), int(scan[i2, 1])

print("Present 1:", scan[i1])
print("Present 2:", scan[i2])


In [ ]:
R = {}
R["present1"] = run_system(H, lambda_i, replace(cfg, seed=seed1), "present", 0, OUT, "Fig4_present1")
R["present2"] = run_system(H, lambda_i, replace(cfg, seed=seed2), "present", 0, OUT, "Fig4_present2")
R["cmrt"]     = run_system(H, lambda_i, cfg, "cmrt", 0, OUT, "Fig4_cmrt")
R["forster"]  = run_system(H, lambda_i, cfg, "forster", 0, OUT, "Fig4_forster")

for key in ("present1", "present2", "cmrt", "forster"):
    r = R[key]
    print(f"{key:8s} Vave = {r['basis']['vave']:.6f} cm^-1")


In [ ]:
PAPER_COLORS_3 = ["#ee2c2a", "#6dbe44", "#2555a6"]  # sites 1, 2, 3

fig, axs = plt.subplots(1, 4, figsize=(14.5, 3.4), sharex=True, sharey=True)
panels = [
    ("present1", "(a) Present 1"),
    ("present2", "(b) Present 2"),
    ("cmrt", "(c) CMRT"),
    ("forster", "(d) Förster"),
]

for ax, (key, title) in zip(axs, panels):
    r = R[key]
    mask = r["time_ps"] > 0
    for i, label in enumerate(("1", "2", "3")):
        ax.plot(
            r["time_ps"][mask], r["populations"][mask, i],
            color=PAPER_COLORS_3[i], linewidth=1.2, label=label,
        )
    ax.set_xscale("log")
    ax.set_xlim(1e-3, 1)
    ax.set_ylim(0, 1)
    ax.set(xlabel="Time (ps)", ylabel="Probability", title=title)
    ax.set_box_aspect(1)
    ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(OUT / "Fig4.pdf", bbox_inches="tight")
plt.show()
